In [ ]:
!pip install -q transformers kagglehub

In [ ]:
import kagglehub

DATA = kagglehub.dataset_download("sattamjaltwaim/ioai-alchemy")
print(f"Dataset path: {DATA}")

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel

random.seed(2026)
np.random.seed(2026)
torch.manual_seed(2026)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load the competition data
train_df = pd.read_csv(f'{DATA}/train.csv')
test_df = pd.read_csv(f'{DATA}/test.csv')
cand_df = pd.read_csv(f'{DATA}/candidates.csv')
candidate_labels = sorted(cand_df['result'].unique().tolist())

print(f"Train: {len(train_df)} rules")
print(f"Test: {len(test_df)} pairs")
print(f"Candidates: {len(candidate_labels)} unique results")

In [ ]:
# Load BERT and its tokenizer -- we only use it to get embeddings, no training needed
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased').to(device).eval()


def get_embedding(text):
    """Run text through BERT and return the CLS token embedding."""
    enc = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=32)
    with torch.no_grad():
        out = model(**{k: v.to(device) for k, v in enc.items()})
    return out.last_hidden_state[:, 0, :].cpu().numpy()

In [ ]:
# Embed every candidate result (the 70 possible answers)
cand_embs = np.vstack([get_embedding(c) for c in candidate_labels])

# Embed every test pair by joining the two items into one string
pair_embs = np.vstack(
    [get_embedding(f"{row['item1']} {row['item2']}") for _, row in test_df.iterrows()]
)

# Normalise so dot product equals cosine similarity
pair_norm = pair_embs / np.linalg.norm(pair_embs, axis=1, keepdims=True)
cand_norm = cand_embs / np.linalg.norm(cand_embs, axis=1, keepdims=True)

# Score matrix: each row is a test pair, each column is a candidate
score_matrix = pair_norm @ cand_norm.T
print(f"Score matrix shape: {score_matrix.shape}")

In [ ]:
# Greedy assignment: for each test pair pick the best unused candidate
col_indices = []
used = set()

for row in score_matrix:
    for c in np.argsort(-row):
        if c not in used:
            col_indices.append(c)
            used.add(c)
            break

In [ ]:
# ============================================
# DO NOT MODIFY -- Submission Generator
# ============================================
predictions = [candidate_labels[c] for c in col_indices]

submission = pd.DataFrame({'Id': test_df['Id'], 'result': predictions})
submission.to_csv('submission.csv', index=False)
print(f"Submission shape: {submission.shape}")
print(submission.head())